# Assignment 8: Build the Intent Dataset for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 02nd November 2026, 23:59**

Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
do the checking, the split, the baseline and the entity count: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Week 7 example) with only 6 messages
per label, so the quality report shows what is still missing. It may not be submitted.

**Steps:** Save a copy of this file in Drive. Replace Steps 1, 2, 3, 6 and 7 with your own, and run
everything from the top (Runtime, then Run all). Repeat Steps 3 and 4 until the report is clean.

---

---

## Setup

**PROVIDED.** Run once. The same spaCy model as the session notebook.

In [9]:
# AITCD001, Week 8 take home: the intent dataset for my own chatbot
import csv
import random
import re
from collections import Counter

import pandas as pd
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:                  # not installed yet: install it once
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("ready: spaCy", spacy.__version__, "| model", nlp.meta["version"])

ready: spaCy 3.8.18 | model 3.8.0


---

## Step 1: The chatbot and its labels

**YOUR DOMAIN.** The chatbot of your Portfolio Project 1. All of its intents, plus `out_of_scope`. Do not add `greeting` or `name` unless your users really ask for something with them.

In [10]:
# STEP 1. YOUR DOMAIN: the chatbot of your Portfolio Project 1, and its labels.
BOT_NAME = "School Canteen Bot"
INTENTS = ["menu", "price", "order", "hours", "status", "human", "out_of_scope"]

TARGET = 30          # messages per label, at least
print(BOT_NAME, "|", len(INTENTS), "labels:", INTENTS)

School Canteen Bot | 7 labels: ['menu', 'price', 'order', 'hours', 'status', 'human', 'out_of_scope']


---

## Step 2: The labelling guideline

**YOUR DOMAIN.** Write the guideline BEFORE labelling. Every time two labels are possible for a message, add or sharpen a rule.

In [18]:
# STEP 2. YOUR DOMAIN: my labelling guideline.
GUIDELINE = [
    "One message, one label. A greeting plus a request is labelled by the request ('hi, how much is rice' is price).",
    "Label the goal, not the words: 'how much for 2 chapati' is price, not order, because the student wants a cost.",
    "Asking what food or drinks exist or are available is menu; asking what an item costs is price, even for 'price list'.",
    "Wanting or buying an item, with or without a quantity, is order, even when 'help' or 'please' appears in the message.",
    "Asking about an order that was already placed (is it ready, where is it) is status, not order.",
    "Asking for a person, staff, the manager or a human is human; a general 'i need help' is also human.",
    "Complaints, payment methods, small talk, a greeting or goodbye alone, other topics and gibberish are out_of_scope.",
    "Keep the student's words: typos, SMS, Kinyarwanda and mixed language, exactly as typed.",
    "Delivery questions (do you deliver, delivery fee) are out_of_scope: the canteen bot does not handle delivery.",
]

for i, rule in enumerate(GUIDELINE, start=1):
    print(i, rule)

1 One message, one label. A greeting plus a request is labelled by the request ('hi, how much is rice' is price).
2 Label the goal, not the words: 'how much for 2 chapati' is price, not order, because the student wants a cost.
3 Asking what food or drinks exist or are available is menu; asking what an item costs is price, even for 'price list'.
4 Wanting or buying an item, with or without a quantity, is order, even when 'help' or 'please' appears in the message.
5 Asking about an order that was already placed (is it ready, where is it) is status, not order.
6 Asking for a person, staff, the manager or a human is human; a general 'i need help' is also human.
7 Complaints, payment methods, small talk, a greeting or goodbye alone, other topics and gibberish are out_of_scope.
8 Keep the student's words: typos, SMS, Kinyarwanda and mixed language, exactly as typed.
9 Delivery questions (do you deliver, delivery fee) are out_of_scope: the canteen bot does not handle delivery.


---

## Step 3: The dataset

**YOUR DOMAIN.** At least 30 messages per label. Sources, in order of value: your chatbot's UNMATCHED log and Step 10 local phrasing from Portfolio Project 1; at least 20 messages from real people you ask (classmates, family, friends, for example on WhatsApp: *how would you ask a bus desk about luggage?*); then your own. Keep every message exactly as it was typed.

In [19]:
# STEP 3. YOUR DOMAIN: the dataset. Each row is (message, label, source).
# source is one of: "rules", "unmatched", "local", "people", "written"
DATA = [
    # ---- menu ----
    ("what do you sell?", "menu", "rules"),
    ("show me the menu", "menu", "rules"),
    ("what do you have", "menu", "rules"),
    ("what food is available", "menu", "rules"),
    ("ibyo mugurisha ni ibihe?", "menu", "local"),
    ("mufite ibiryo bihe uyu munsi?", "menu", "local"),
    ("menu ni iyihe?", "menu", "local"),
    ("ndashaka kureba menu", "menu", "local"),
    ("hari ibiryo biriho?", "menu", "local"),
    ("muraho, mufite iki?", "menu", "local"),
    ("whats on the menu today", "menu", "written"),
    ("wat do u sell", "menu", "written"),
    ("do you have any drinks", "menu", "written"),
    ("list the food pls", "menu", "written"),
    ("what can i eat there", "menu", "written"),
    ("do you sell juice", "menu", "written"),
    ("is there anything to eat", "menu", "written"),
    ("menu please", "menu", "written"),
    ("wat food do u hv", "menu", "written"),
    ("can i see what you are selling", "menu", "written"),
    ("what is available today", "menu", "written"),
    ("do you sell chips", "menu", "written"),
    ("which meals do you have", "menu", "written"),
    ("send me the menu", "menu", "written"),
    ("what are the options for lunch", "menu", "written"),
    ("show me wat u have", "menu", "written"),
    ("any food left", "menu", "written"),
    ("do you have rice and beans", "menu", "written"),
    ("what is for lunch today", "menu", "written"),
    ("hello, what food do you have", "menu", "written"),
    # ---- price ----
    ("how much is chapati", "price", "rules"),
    ("hi, how much is rice", "price", "rules"),
    ("what is the price of beans", "price", "rules"),
    ("how much does juice cost", "price", "rules"),
    ("umuceri ni angahe?", "price", "local"),
    ("ibishyimbo ni bangahe?", "price", "local"),
    ("chapati igura angahe?", "price", "local"),
    ("igiciro cya juice ni ikihe?", "price", "local"),
    ("chips ni angahe?", "price", "local"),
    ("muraho, umuceri ni angahe", "price", "local"),
    ("price of chips", "price", "written"),
    ("how much for 2 chapati", "price", "written"),
    ("wat is the cost of juice", "price", "written"),
    ("hw mch is rice", "price", "written"),
    ("rice price?", "price", "written"),
    ("how much do beans cost", "price", "written"),
    ("is juice 500", "price", "written"),
    ("how much r the chips", "price", "written"),
    ("cost of a chapati", "price", "written"),
    ("what do i pay for beans and rice", "price", "written"),
    ("how much is everything", "price", "written"),
    ("price list please", "price", "written"),
    ("is chapati cheaper than chips", "price", "written"),
    ("how much is one juice", "price", "written"),
    ("tell me the price of rice", "price", "written"),
    ("chips cost how much", "price", "written"),
    ("what is the cheapest item", "price", "written"),
    ("how much 4 rice n beans", "price", "written"),
    ("pls how much is chapati", "price", "written"),
    ("do chips cost 800", "price", "written"),
    # ---- order ----
    ("I want 2 chapatis and 1 juice", "order", "rules"),
    ("Hello, I want 2 beans", "order", "rules"),
    ("can you help me order chips", "order", "rules"),
    ("give me rice", "order", "rules"),
    ("i ll take 3 chips", "order", "rules"),
    ("ndashaka umuceri", "order", "local"),
    ("muraho, ndashaka umuceri", "order", "local"),
    ("mumpe ibishyimbo bibiri", "order", "local"),
    ("ndashaka chapati ebyiri", "order", "local"),
    ("mpa juice imwe", "order", "local"),
    ("ndashaka kugura chips", "order", "local"),
    ("get me 2 chips", "order", "written"),
    ("i would like some rice", "order", "written"),
    ("one rice and beans pls", "order", "written"),
    ("can i order rice", "order", "written"),
    ("2 chapati pls", "order", "written"),
    ("i wanna buy a juice", "order", "written"),
    ("let me have beans", "order", "written"),
    ("i need 3 chapatis", "order", "written"),
    ("order 1 chips and 1 juice", "order", "written"),
    ("can i get rice n beans", "order", "written"),
    ("put me down for 2 juice", "order", "written"),
    ("id like to order lunch", "order", "written"),
    ("bring me chips", "order", "written"),
    ("i want rice", "order", "written"),
    ("1 beans 1 chapati", "order", "written"),
    ("give me two juices", "order", "written"),
    ("hi i want to order", "order", "written"),
    ("i want to place an order", "order", "written"),
    ("can i have chips", "order", "written"),
    ("buy 2 rice for me", "order", "written"),
    # ---- hours ----
    ("when do you open?", "hours", "rules"),
    ("what time do you stop selling?", "hours", "unmatched"),
    ("what are your opening hours", "hours", "rules"),
    ("are you open on friday", "hours", "rules"),
    ("mufungura saa ngahe?", "hours", "local"),
    ("mufunga saa ngahe?", "hours", "local"),
    ("mukora ku wa gatandatu?", "hours", "local"),
    ("kantine ifungura ryari?", "hours", "local"),
    ("mufunguye none?", "hours", "local"),
    ("muraho, mukora kugeza saa ngahe?", "hours", "local"),
    ("wen do u close", "hours", "written"),
    ("are you open on saturday", "hours", "written"),
    ("opening time pls", "hours", "written"),
    ("what time does the canteen open", "hours", "written"),
    ("do you work on sunday", "hours", "written"),
    ("is the canteen open today", "hours", "written"),
    ("closing time?", "hours", "written"),
    ("open on monday?", "hours", "written"),
    ("until what time are you open", "hours", "written"),
    ("what days are you open", "hours", "written"),
    ("are you open at 3pm", "hours", "written"),
    ("whats ur schedule", "hours", "written"),
    ("do you open on weekends", "hours", "written"),
    ("from what time do you serve", "hours", "written"),
    ("hours?", "hours", "written"),
    ("when does lunch start", "hours", "written"),
    ("is it open now", "hours", "written"),
    ("open till 16:00?", "hours", "written"),
    ("what time do you open on wednesday", "hours", "written"),
    ("are you open on thursday", "hours", "written"),
    # ---- status ----
    ("where is my order", "status", "rules"),
    ("is it ready", "status", "rules"),
    ("order status", "status", "rules"),
    ("ibyo nasabye byarangiye?", "status", "local"),
    ("order yanjye iri he?", "status", "local"),
    ("order yanjye yarangiye?", "status", "local"),
    ("ese ibyo naguze bigezehe?", "status", "local"),
    ("muraho, order yanjye iri he", "status", "local"),
    ("is my food ready", "status", "written"),
    ("how long for my order", "status", "written"),
    ("wheres my food", "status", "written"),
    ("when will my chapati be ready", "status", "written"),
    ("check my order", "status", "written"),
    ("did u get my order", "status", "written"),
    ("is my lunch ready yet", "status", "written"),
    ("how many minutes for my order", "status", "written"),
    ("status of my order", "status", "written"),
    ("any update on my food", "status", "written"),
    ("is my rice ready", "status", "written"),
    ("can i collect my order", "status", "written"),
    ("has my order been made", "status", "written"),
    ("wat is happening with my order", "status", "written"),
    ("when can i pick up my food", "status", "written"),
    ("order ready?", "status", "written"),
    ("i ordered 10 minutes ago, where is it", "status", "written"),
    ("where is my juice", "status", "written"),
    ("is my order done", "status", "written"),
    ("hi, is my food ready", "status", "written"),
    ("how far is my order", "status", "written"),
    ("my order, is it ready now", "status", "written"),
    # ---- human ----
      ("what time it opens?", "hours", "people"),
      ("what food do they sell?", "menu", "people"),
      ("I want 3 bottles  of Yoghurt on Friday 2:00PM .", "order", "people"),
      ("What tym u open?", "hours", "people"),
      ("Do you do delivery and how much is delivery to Gisozi", "out_of_scope", "people"),
      ("Hi, what food do you have today?", "menu", "people"),
      ("How much is a plate of rice?", "price", "people"),
      ("I want to order rice and beans, please.", "order", "people"),
      ("What time does the canteen open?", "hours", "people"),
      ("Hey, is my food ready?", "status", "people"),
      ("How much is rice?", "price", "people"),
      ("Hi, I’d like to order rice and beans, please.", "order", "people"),
      ("Is my order ready yet?", "status", "people"),
      ("What Are the Food Do you have ?", "menu", "people"),
      ("What about prices?", "price", "people"),
    # ---- out_of_scope ----
    ("asdfgh", "out_of_scope", "unmatched"),
    ("murakoze cyane", "out_of_scope", "local"),
    ("mwiriwe", "out_of_scope", "local"),
    ("amakuru?", "out_of_scope", "local"),
    ("nshobora kwishyura na momo?", "out_of_scope", "local"),
    ("ibiryo byari bikonje", "out_of_scope", "local"),
    ("who won the match", "out_of_scope", "written"),
    ("can i pay with momo", "out_of_scope", "written"),
    ("the food was cold yesterday", "out_of_scope", "written"),
    ("do you have a job for students", "out_of_scope", "written"),
    ("tell me a joke", "out_of_scope", "written"),
    ("what is the weather today", "out_of_scope", "written"),
    ("thanks", "out_of_scope", "written"),
    ("hello", "out_of_scope", "written"),
    ("bye", "out_of_scope", "written"),
    ("good morning", "out_of_scope", "written"),
    ("ok", "out_of_scope", "written"),
    ("the chapati was burnt", "out_of_scope", "written"),
    ("how do i cook rice", "out_of_scope", "written"),
    ("where is the library", "out_of_scope", "written"),
    ("who is the headteacher", "out_of_scope", "written"),
    ("my phone is lost", "out_of_scope", "written"),
    ("i love this bot", "out_of_scope", "written"),
    ("what is your name", "out_of_scope", "written"),
    ("the juice was sour", "out_of_scope", "written"),
    ("can i pay later", "out_of_scope", "written"),
    ("do you give credit", "out_of_scope", "written"),
    ("sing a song", "out_of_scope", "written"),
    ("what is 2+2", "out_of_scope", "written"),
    ("qwerty", "out_of_scope", "written"),
    ("lol", "out_of_scope", "written"),
    ("the queue is too long", "out_of_scope", "written"),
]

# Real messages from people I asked (WhatsApp). Paste them here exactly as typed.
# Format: ("message", "label", "people"),   No names, no phone numbers.
PEOPLE = [
    # ("how much is juice at the canteen", "price", "people"),
]

DATA = DATA + PEOPLE
print(len(DATA), "rows")

198 rows


---

## Step 4: The quality report

**PROVIDED.** Read every line. Fix the data in Step 3 until every label is ok, there are no duplicates and no conflicts, and the balance is 1.5 or less. Then run Step 3 and Step 4 again.

In [13]:
# STEP 4. PROVIDED: the quality report. Read it, fix the data in Step 3, run again.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


per_label = Counter()
per_source = Counter()
unknown = []
for text, label, source in DATA:
    per_label[label] += 1
    per_source[source] += 1
    if label not in INTENTS:
        unknown.append((text, label))

print("Per label:")
for label in INTENTS:
    n = per_label[label]
    note = "ok" if n >= TARGET else f"needs more, target {TARGET}"
    print(f"   {label:14}{n:4}   {note}")
print("Per source:", dict(per_source))
print("people:", per_source["people"], "of at least 20")

seen = {}
duplicates = []
conflicts = []
for text, label, source in DATA:
    key = clean(text)
    if key in seen:
        duplicates.append(text)
        if seen[key] != label:
            conflicts.append(text)
    seen[key] = label

print("Labels not in INTENTS:", unknown)
print("Duplicates:", duplicates)
print("Conflicting labels:", conflicts)
counts = [per_label[label] for label in INTENTS]
print("Balance, largest / smallest:", round(max(counts) / max(1, min(counts)), 2))

Per label:
   menu            30   ok
   price           30   ok
   order           31   ok
   hours           30   ok
   status          30   ok
   human           30   ok
   out_of_scope    32   ok
Per source: {'rules': 22, 'local': 39, 'written': 150, 'unmatched': 2}
people: 0 of at least 20
Labels not in INTENTS: []
Duplicates: []
Conflicting labels: []
Balance, largest / smallest: 1.07


---

## Step 5: The split, and the CSV file

**PROVIDED.** 20% of each label becomes the test set. The file `intents.csv` appears in the Files panel on the left: download it, and commit it to GitHub.

In [14]:
# STEP 5. PROVIDED: the 80/20 split, intent by intent, with a fixed seed; then the CSV.
rng = random.Random(8)

by_label = {}
for row in DATA:
    if row[1] not in by_label:
        by_label[row[1]] = []
    by_label[row[1]].append(row)

TRAIN = []
TEST = []
for label in sorted(by_label):
    rows = by_label[label][:]
    rng.shuffle(rows)
    cut = len(rows) // 5
    TEST.extend(rows[:cut])
    TRAIN.extend(rows[cut:])

with open("intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, label, source in TRAIN:
        writer.writerow([text, label, source, "train"])
    for text, label, source in TEST:
        writer.writerow([text, label, source, "test"])

df = pd.read_csv("intents.csv")
print(len(TRAIN), "training,", len(TEST), "test, saved to intents.csv")
print(pd.crosstab(df["intent"], df["split"]))

171 training, 42 test, saved to intents.csv
split         test  train
intent                   
hours            6     24
human            6     24
menu             6     24
order            6     25
out_of_scope     6     26
price            6     24
status           6     24


---

## Step 6: The baseline: your rules on your test set

**YOUR DOMAIN.** Paste the rule table of your own chatbot. The baseline is the number your classifier must beat in Week 9. Every line it prints is a message the rules get wrong.

In [15]:
# STEP 6. YOUR DOMAIN: the RULES of my Portfolio Project 1 canteen bot (final version).
RULES = [
    ("status", r"\b(where is my order|order status|status|is it ready)\b"),
    ("price",  r"\b(how much|price|cost)\b"),
    ("menu",   r"\b(menu|sell|available|what do you have|food)\b"),
    ("order",  r"\b(order|want|buy|give me|get me|i ?ll take|ndashaka)\b"),
    ("hours",  r"\b(open|close|closing|hours|time)\b"),
    ("human",  r"\b(person|human|staff|agent|help)\b"),
]
print(len(RULES), "rules")

6 rules


In [16]:
# STEP 6, continued. PROVIDED: the baseline, your rules on your test set.
def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


right = 0
for text, label, source in TEST:
    guess = rule_label(text)
    if guess == label:
        right += 1
    else:
        print(f"   {text!r}: labelled {label}, rules say {guess}")
print(f"Baseline: {right} of {len(TEST)} test messages right")

   'whats ur schedule': labelled hours, rules say out_of_scope
   'mufunga saa ngahe?': labelled hours, rules say out_of_scope
   'mufunguye none?': labelled hours, rules say out_of_scope
   'let me speak to someone': labelled human, rules say out_of_scope
   'give me a real person': labelled human, rules say order
   'i dont want a bot': labelled human, rules say order
   'nkeneye ubufasha': labelled human, rules say out_of_scope
   'hari umuntu wo kumfasha?': labelled human, rules say out_of_scope
   'do you have any drinks': labelled menu, rules say out_of_scope
   'can i see what you are selling': labelled menu, rules say out_of_scope
   'show me wat u have': labelled menu, rules say out_of_scope
   'can i get rice n beans': labelled order, rules say out_of_scope
   'i would like some rice': labelled order, rules say out_of_scope
   'put me down for 2 juice': labelled order, rules say out_of_scope
   'ibishyimbo ni bangahe?': labelled price, rules say out_of_scope
   'chips ni anga

---

## Step 7: The entities of your domain

**YOUR DOMAIN.** All entity labels of your own, each with the closed list of names your users mention: places, products, services, days. The provided part counts how often they appear in your dataset.

In [20]:
# STEP 7. YOUR DOMAIN: the entities of my domain. At least 2 labels of my own.
ENTITY_LISTS = {
    "ITEM": ["rice", "beans", "chapati", "chapatis", "chips", "juice", "juices", "umuceri", "ibishyimbo", "yoghurt"],
    "DAY": ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "today"],
}

patterns = []
for label in ENTITY_LISTS:
    for name in ENTITY_LISTS[label]:
        patterns.append({"label": label, "pattern": [{"LOWER": name}]})

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(patterns)

# PROVIDED: how many of your messages contain each entity, and five examples
found = Counter()
examples = []
for text, label, source in DATA:
    ents = [(e.text, e.label_) for e in nlp(text.lower()).ents]
    for e in ents:
        found[e[1]] += 1
    if ents and len(examples) < 5:
        examples.append((text, ents))
print("Entities found in the dataset:", dict(found))
for text, ents in examples:
    print("  ", text, "->", ents)

Entities found in the dataset: {'ORG': 3, 'PERSON': 9, 'DAY': 13, 'ITEM': 80, 'GPE': 1, 'CARDINAL': 22, 'TIME': 4, 'DATE': 1}
   ibyo mugurisha ni ibihe? -> [('mugurisha ni ibihe', 'ORG')]
   ndashaka kureba menu -> [('ndashaka kureba menu', 'PERSON')]
   hari ibiryo biriho? -> [('hari', 'ORG')]
   whats on the menu today -> [('today', 'DAY')]
   do you sell juice -> [('juice', 'ITEM')]


---

## Step 8: Publish and submit

1. In the repository of your Portfolio Project 1, create a folder named `dataset`.
2. Upload this notebook (File, then Download, then .ipynb) and `intents.csv` into it.
3. Add `dataset/README.md` from `README_template_dataset.md`, every section filled in.
4. Open the folder in a private browsing window. Paste its URL in Google Classroom and click
   **Turn in**, by **Monday 02 November 2026, 23:59**.

Never put real names, phone numbers or ID numbers in the dataset: it is public.